In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib
from random import random
from model import DINOCell
from pipeline import get_pipeline
from PIL import Image
import torch
import numpy as np
import os
import sys

In [2]:
def convert_label_to_rainbow(label):
    label_rainbow = np.zeros((label.shape[0], label.shape[1], 3), dtype=np.uint8)
    for cell in np.unique(label):
        if cell == 0:
            continue #background
        label_rainbow[label == cell] = np.clip(np.random.rand(3) * 255, a_min=10, a_max=255)

    return label_rainbow

In [ ]:
num_epochs = 40 #40, 80 #CHANGE
patch_size = 8 #8, 14 CHANGE
crop_size = 512 #512, 256 CHANGE
finetune_vision = True 
lower_lr_encoder = False # CHANGE 
delayed_encoder_unfreezing = False 
hyperparameter_type = 'hyperparam_5' #original, cellpose_sam, gpt_suggested, hyperparam_4, hyperparam_5, hyperparam_6 #CHANGE
dropout_in_encoder = True
use_ignore_masks = True
ignore_all = True
use_advanced_augmentations = False #CHANGE
include_silver_data = False
use_dino_weights = True # True, False CHANGE
resume_training = True #CHANGE
dataset_root_path = '../../Dataset/Full_Segmentation_Dataset'
dataset_type = 'pc_bf_non_tissue_whole_cell_only' #full_dataset_list, fluorescent, pc_bf_non_tissue_whole_cell_only, tissue CHANGE
dino_type = f'domain_adaption_patch_size_{patch_size}' #CHANGE
# dino_type = f'random_init_patch_size_{patch_size}'
# dino_type = f'pretrained_but_not_domain_adapted'
# dino_model = f'../pretrained_dino_models/dino_{dino_type}_checkpoint_19999.pth' # CHANGE
dino_model = f'../pretrained_dino_models/dino_{dino_type}_checkpoint_24799.pth'
# dino_model = f'../pretrained_dino_models/dino_{dino_type}_checkpoint_99999.pth'
if dino_type == f'random_init_patch_size_{patch_size}':
    dino_model = '../pretrained_dino_models/dino_random_init_patch_size_14_checkpoint_34399.pth'
elif dino_type == 'pretrained_but_not_domain_adapted':
    dino_model = f'../pretrained_dino_models/dino_{dino_type}.pth'
decoder_type = 'upsample' # upsample, convolutional, SAM
objective_type = 'flows' #'distance_map', 'directional_distance_map', 'flows', 'direct_mask_prediction', 'direct_mask_prediction_independent_encoding' #CHANGE
ignore_mask_weight = 0.05 #0.2, 0.02, 0.05 CHANGE
silver_data_weight = 0.3 #0.1, 0.3, 0.5, 1.0
log_modifier = f'piw_{str(int(ignore_mask_weight * 100))}_isd_{str(include_silver_data)}_hpt_{hyperparameter_type}_die_{dropout_in_encoder}_ftv_{str(finetune_vision)}_llre_{str(lower_lr_encoder)}_deu_{str(delayed_encoder_unfreezing)}_cs_{crop_size}_sdw_{str(int(silver_data_weight * 100))}_ne_{num_epochs}'
# log_modifier += f'_udw_{use_dino_weights}'
# log_modifier += '_longer_domain_adaption' #CHANGE
# log_modifier += '_advanced_augmentations'
#piw = partial_ignore_weight, isd = include_silver_data, hpt = hyper_parameter_type, die = dropout_in_encoder, ftv = finetune_vision, llre = lower_lr_encoder, deu = delayed_encoder_unfreezing, cs = crop_size, sdw = silver_data_weight, ne = num_epochs,


#hyperparameter setup
if hyperparameter_type == 'original':
    lr = 1e-5
    weight_decay = 1e-3
    drop_rate = 0.0
elif hyperparameter_type == 'cellpose_sam':
    lr = 5e-5
    weight_decay = 0.1
    drop_rate = 0.4
elif hyperparameter_type == 'gpt_suggested':
    lr = 1e-5
    weight_decay = 1e-4
    drop_rate = 0.1
elif hyperparameter_type == 'hyperparam_4':
    lr = 3e-5
    weight_decay = 1e-4
    drop_rate = 0.02
elif hyperparameter_type == 'hyperparam_5':
    lr = 2e-5
    weight_decay = 1e-4
    drop_rate = 0.05
elif hyperparameter_type == 'hyperparam_6':
    lr = 1e-5
    weight_decay = 1e-4
    drop_rate = 0.05

# Load the model
sam_model = 'facebook/sam-vit-base' #deprecated but leaving for now
if objective_type == 'direct_mask_prediction':
    flows_model_modifier = f'piw_{str(int(ignore_mask_weight * 100))}_isd_{str(include_silver_data)}_hpt_{hyperparameter_type}_die_{dropout_in_encoder}_ftv_{str(finetune_vision)}_llre_{str(lower_lr_encoder)}_deu_{str(delayed_encoder_unfreezing)}_cs_{crop_size}_sdw_{str(int(silver_data_weight * 100))}_ne_{num_epochs}'
    flows_model = f'../models/DINOCell_{dataset_type}_{dino_type}_{decoder_type}_flows_{flows_model_modifier}_train_checkpoint.pth'
else:
    flows_model = None
print('loading model...')
model = DINOCell(dino_model, sam_model, flows_model, decoder_type, objective_type, use_dino_weights, patch_size, feat_size=64, crop_size=crop_size, drop_rate=drop_rate, dropout_in_encoder=dropout_in_encoder, finetune_vision=finetune_vision, finetune_decoder=True, finetune_prediction_head=True) 
# model_save_path = f'../models/DINOCell_{dataset_type}_{dino_type}_{decoder_type}_{objective_type}_{log_modifier}.pt'
model_save_path = f'../models/BEST_MODEL_DINOCell_{dataset_type}_{dino_type}_{decoder_type}_{objective_type}_{log_modifier}.pt'
model.load_state_dict(torch.load(model_save_path, map_location=torch.device('cpu')))

#device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

pipeline = get_pipeline(objective_type, model, device, crop_size=crop_size, use_advanced_augmentations=use_advanced_augmentations, overlap_size=crop_size//4)

In [ ]:
img_path = ''
image = np.array(Image.open(img_path))

pred_labels, flows, img = pipeline.run(image, return_raw_preds=True)

#~~~~~~~~~~~~~~~~~~~~~~
#   VISUALIZATION
#~~~~~~~~~~~~~~~~~~~~~~

mask_preds_rgb = convert_label_to_rainbow(pred_labels)
mask_preds_rgb[pred_labels == 0] = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)[mask_preds == 0]


fig, ax = plt.subplots(1, 2, figsize=(10, 10))

#remove axes
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[1].set_xticks([])
ax[1].set_yticks([])

#set 1 in borders
ax[0].spines['top'].set_visible(False)
ax[0].spines['right'].set_visible(False)
ax[0].spines['bottom'].set_visible(False)
ax[0].spines['left'].set_visible(False)
ax[1].spines['top'].set_visible(False)
ax[1].spines['right'].set_visible(False)
ax[1].spines['bottom'].set_visible(False)
ax[1].spines['left'].set_visible(False)

ax[0].imshow(image, cmap='gray')
ax[1].imshow(image, cmap='gray')
ax[1].imshow(mask_preds_rgb, cmap='gray', alpha=0.7)

plt.show()